## Preamble

In [ ]:
import os
import numpy as np
#from scipy import signal
import matplotlib.pyplot as plt

def read(filename,n=(nx,nz)):
    return np.fromfile(filename,dtype='float32').reshape(n).T

nr=100
nt=600
def read_su(filename,n=(nr,nt)):
    data=read(filename,n=(n[0],int(60+n[1])))
    return data[60:,:]

nsnap, nxsnap, nzsnap = 51, 249, 249
def read_snap(filename,n=(nsnap,nxsnap,nzsnap),i=17):
    tmp=np.fromfile(filename,dtype='float32').reshape(n).T
    return tmp[24:24+nz,24:24+nx,i]

def imshow(data,perc=None,clip=None,clipmin=None,extent=None,cmap='viridis',title=None,grid=True):
    
    if perc==None:
        clipp=[np.amin(data),np.amax(data)]
    else:
        tmp=np.percentile(np.abs(data),q=perc)
        clipp=[-tmp,tmp]
    
    if clip!=None: clipp=clip
    
    if clipmin!=None: clipp[0]=clipmin
    
    plt.imshow(data,vmin=clipp[0],vmax=clipp[1],extent=extent,cmap=cmap,aspect='auto')
    plt.colorbar(location='right')
    plt.grid(visible=grid, axis='both', which='both', color='w', linestyle='--',linewidth=0.5)
    #plt.xlabel(labels[0]); plt.ylabel(labels[1])
    if title!=None: plt.title(title)

nz=201; nx=201

!makevel nz=100 nx=$nx v000=1000 > c1
!makevel nz=101 nx=$nx v000=1800 > c2
!cat c1 c2 > tmp && transp < tmp n1=$nz > simple
!rm c1 c2 tmp

!makevel nx=201 nz=201 v000=2000 > model

In [9]:
simple=1000*np.ones((nz,nx))
simple[101:,:]=1800
simple.T.astype('float32').tofile('simple')

model=2000*np.ones((nz,nx))
model.T.astype('float32').tofile('model')

In [ ]:
imshow(read('simple'))

In [4]:
#!cd ../../; sed -i 's/WaveEq=DAS/WaveEq=SH/' modules.inc
#!make
!make cleanall; make
#!make

#System
(cd ../../Modules/System; make clean)
make[1]: Entering directory '/home/joey/Codes/GitHub/SeisJIMU/Modules/System'
rm m_either.o m_string.o m_mpienv.o m_message.o m_arrayop.o m_setup.o m_sysio.o m_checkpoint.o m_suformat.o m_System.o

rm ../../mod/m_either.mod ../../mod/m_string.mod ../../mod/m_mpienv.mod ../../mod/m_message.mod ../../mod/m_arrayop.mod ../../mod/m_setup.mod ../../mod/m_sysio.mod ../../mod/m_checkpoint.mod ../../mod/m_suformat.mod ../../mod/m_System.mod ../../mod/m_system.mod
rm: cannot remove '../../mod/m_System.mod': No such file or directory
make[1]: [Makefile:25: clean] Error 1 (ignored)

make[1]: Leaving directory '/home/joey/Codes/GitHub/SeisJIMU/Modules/System'
#Etc
(cd ../../Modules/Etc; make clean)
make[1]: Entering directory '/home/joey/Codes/GitHub/SeisJIMU/Modules/Etc'
rm sgtsv.o m_math.o singleton.o

rm ../../mod/m_math.mod ../../mod/singleton.mod

make[1]: Leaving directory '/home/joey/Codes/GitHub/SeisJIMU/Modules/Etc'
#Signal
(cd ../../Modules/S

In [27]:
!cat setup_default

MODEL_SIZE              '201 201 1'
MODEL_SPACING           '10 10 1'
FILE_MODEL              'simple'
MODEL_ATTRIBUTES         vs  #'vp rho'

IS_FREESURFACE          T
IF_HICKS        T

ACQUI_GEOMETRY          spread
#FS                      '0 500 500'
FS                       '2 500 500'
#FS                      '10 500 500'
#FS                      '40 500 500'
#FS                      '500 500 500'

#FR                      '0 10  0'
FR                       '2 10  0'
#FR                      '10 10  0'
#FR                      '40 10  0'
#FR                      '40 500  0'
#FR                      '500 10  0'

DR                      '0  20 0'
#NR                      1
NR                      100

SCOMP                   vy
RCOMP                   vy

WAVELET_TYPE            ricker
#T0                      0.5                                                                 
#FILE_WAVELET            'fricker_dt6000.su' 

#IF_BLOOM        F


IF_USE_RANDOM   F

NT           1500

### vy-vy

In [21]:
!cp setup_default setup
!echo 'IS_FREESURFACE F' >> setup
!../../exe/AdjointTest  setup > out

!tail -10 out

 LHS = <  v|Lu> =    4.3010638665925076E-017
 RHS = <Lᴴv| u> =    4.3010640323094165E-017
 relative difference =    3.8529281981573440E-006  %
   SeisJIMU has finished the job   
 System date: 02/19/2026
System time: 06:25:38
System timezone: +08:00
                        


In [22]:
!cp setup_default setup
#!echo 'IS_FREESURFACE F' >> setup
!../../exe/AdjointTest  setup > out

!tail -10 out

 LHS = <  v|Lu> =    5.1560433113973603E-016
 RHS = <Lᴴv| u> =    5.1560432596456867E-016
 relative difference =    1.0037090535150066E-006  %
   SeisJIMU has finished the job   
 System date: 02/19/2026
System time: 06:25:53
System timezone: +08:00
                        


### szy-szy

In [23]:
!cp setup_default setup
!echo 'IS_FREESURFACE F' >> setup
!echo 'SCOMP        szy' >> setup
!echo 'RCOMP        szy' >> setup
!../../exe/AdjointTest  setup > out

!tail -10 out

 LHS = <  v|Lu> =    112273527.63739222     
 RHS = <Lᴴv| u> =    112273520.71212372     
 relative difference =    6.1682113733967038E-006  %
   SeisJIMU has finished the job   
 System date: 02/19/2026
System time: 06:28:07
System timezone: +08:00
                        


In [24]:
!cp setup_default setup
#!echo 'IS_FREESURFACE F' >> setup
!echo 'SCOMP        szy' >> setup
!echo 'RCOMP        szy' >> setup
!../../exe/AdjointTest  setup > out

!tail -10 out

 LHS = <  v|Lu> =    7246501.2743126666     
 RHS = <Lᴴv| u> =    7246499.1188590946     
 relative difference =    2.9744748402796389E-005  %
   SeisJIMU has finished the job   
 System date: 02/19/2026
System time: 06:28:26
System timezone: +08:00
                        


### vy-szy

In [25]:
!cp setup_default setup
!echo 'IS_FREESURFACE F' >> setup
!echo 'RCOMP        szy' >> setup
!../../exe/AdjointTest  setup > out

!tail -10 out

 LHS = <  v|Lu> =    2.6929302479679459E-007
 RHS = <Lᴴv| u> =    2.6934033429270430E-007
 relative difference =    1.7564950319807034E-002  %
   SeisJIMU has finished the job   
 System date: 02/19/2026
System time: 06:29:26
System timezone: +08:00
                        


In [43]:
!cp setup_default setup
!echo 'RCOMP        szy' >> setup
!../../exe/AdjointTest  setup > out

!tail -10 out

 LHS = <  v|Lu> =    1.1690600950103087E-004
 RHS = <Lᴴv| u> =    1.0526854010194545E-004
 relative difference =    9.9545519077723750       %
   SeisJIMU has finished the job   
 System date: 02/19/2026
System time: 06:47:29
System timezone: +08:00
                        


#### .this is off..

### vy-sxy

In [37]:
!cp setup_default setup
!echo 'IS_FREESURFACE F' >> setup
!echo 'RCOMP        sxy' >> setup
!../../exe/AdjointTest  setup > out

!tail -10 out

 LHS = <  v|Lu> =    7.5240051095245918E-005
 RHS = <Lᴴv| u> =    7.5240047671244005E-005
 relative difference =    4.5507703188931241E-006  %
   SeisJIMU has finished the job   
 System date: 02/19/2026
System time: 06:45:24
System timezone: +08:00
                        


In [44]:
!cp setup_default setup
!echo 'RCOMP        sxy' >> setup
!../../exe/AdjointTest  setup > out

!tail -10 out

 LHS = <  v|Lu> =    1.0187946626285000E-004
 RHS = <Lᴴv| u> =    8.7423783226945953E-005
 relative difference =    14.189005465152565       %
   SeisJIMU has finished the job   
 System date: 02/19/2026
System time: 06:47:53
System timezone: +08:00
                        


#### .this is off..

### szy-vy

In [39]:
!cp setup_default setup
!echo 'IS_FREESURFACE F' >> setup
!echo 'SCOMP        szy' >> setup
!../../exe/AdjointTest  setup > out

!tail -10 out

 LHS = <  v|Lu> =    2.6938790805016085E-007
 RHS = <Lᴴv| u> =    2.6934021557791569E-007
 relative difference =    1.7704013736311735E-002  %
   SeisJIMU has finished the job   
 System date: 02/19/2026
System time: 06:45:49
System timezone: +08:00
                        


In [45]:
!cp setup_default setup
!echo 'SCOMP        szy' >> setup
!../../exe/AdjointTest  setup > out

!tail -10 out

 LHS = <  v|Lu> =    9.4804880329375053E-005
 RHS = <Lᴴv| u> =    1.0527129372805723E-004
 relative difference =    9.9423242823628737       %
   SeisJIMU has finished the job   
 System date: 02/19/2026
System time: 06:48:33
System timezone: +08:00
                        


#### .this is off..

### sxy-vy

In [41]:
!cp setup_default setup
!echo 'IS_FREESURFACE F' >> setup
!echo 'SCOMP        sxy' >> setup
!../../exe/AdjointTest  setup > out

!tail -10 out

 LHS = <  v|Lu> =    7.5240066791868468E-005
 RHS = <Lᴴv| u> =    7.5240061070978452E-005
 relative difference =    7.6035153342964853E-006  %
   SeisJIMU has finished the job   
 System date: 02/19/2026
System time: 06:46:21
System timezone: +08:00
                        


In [46]:
!cp setup_default setup
!echo 'SCOMP        sxy' >> setup
!../../exe/AdjointTest  setup > out

!tail -10 out

 LHS = <  v|Lu> =    7.5027530728464687E-005
 RHS = <Lᴴv| u> =    8.7424327469132548E-005
 relative difference =    14.180031004579218       %
   SeisJIMU has finished the job   
 System date: 02/19/2026
System time: 06:48:56
System timezone: +08:00
                        


#### .this is off..